In [1]:
%pip install transformers bitsandbytes accelerate torch kernels

  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached kernels-0.13.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.24.2-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
  Using cached tomlkit-0.14.0-py3-none-any.whl.metadata (2.8 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5

In [8]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(torch.cuda.memory_summary())


CUDA available: True
GPU name: NVIDIA H200 NVL
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |  62341 MiB |  71421 MiB | 276175 GiB | 276114 GiB |
|       from large pool |  62314 MiB |  71395 MiB | 246269 GiB | 246208 GiB |
|       from small pool |     27 MiB |     35 MiB |  29906 GiB |  29906 GiB |
|---------------------------------------------------------------------------|
| Active memory         |  62341 MiB |  71421 MiB | 276175 GiB | 276114 GiB |
|       from larg

In [1]:
import os
print(os.getpid())

3415


In [2]:
!nvidia-smi

Fri Apr 24 23:48:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA H200 NVL                On  |   00000000:00:05.0 Off |                    0 |
| N/A   52C    P0            372W /  600W |  131247MiB / 143771MiB |    100%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [3]:
import os

folder_path = "tables/"

folder_path_LLM_statements = "GPT/b.LLM_Inferences"

folder_path_python_code = "GPT/c.checking_statements"

folder_path_python_output_checking_statements = "GPT/d.checking_statements_output"

In [4]:
#initalizing the model with 4 bit quantization

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "openai/gpt-oss-120b"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype="auto", 
    low_cpu_mem_usage=True,
)

Fetching 42 files:   0%|          | 0/42 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/615 [00:00<?, ?it/s]

In [5]:
from transformers import pipeline
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

In [6]:
import re

def generate(b):
    
    # List all CSV files in the folder
    batch_start = b
    csv_files = [f for f in os.listdir(folder_path) 
                 if f.startswith("table_") and f.endswith(".csv") 
                 and batch_start <= int(f.split('_')[1].split('.')[0]) < batch_start + 10]
    csv_files.sort()  # optional: ensure consistent order
    
    for csv_file in csv_files:
        full_path = os.path.join(folder_path, csv_file)
    
        # Read the table
        with open(full_path, "r", encoding="utf-8") as f:
            lines = f.read().splitlines()
    
        if not lines:
            print(f"{csv_file} is empty, skipping")
            continue
    
        # Extract header and rows
        header = lines[0]
        rows = lines[1:]
    
        print(f"Processing {csv_file}: {len(rows)} rows")
    
        prompt1 = [
        {
            "role": "system",
            "content": """You are an expert data analyst and logician.
    
    You may think silently, but your visible output must be ONLY valid JSON.
    
    Return exactly one JSON object with this schema:
    {"statements": ["...", "..."]}
    
    Rules:
    - 5 to 10 statements
    - each statement must be factually true from the table
    - non-trivial and high-information
    - natural language
    - no markdown
    - no explanation
    - no preamble
    - no trailing text
    - do not wrap the JSON in code fences
    """
        },
        {
            "role": "user",
            "content": f"""
            Your task is to generate natural language statements that describe patterns, relationships, and notable observations in this data. Generate as many **distinct, non-redundant** statements as the data supports.
    
    REQUIREMENTS:
    
    1. **Factual Accuracy**: Each statement must be verifiable against the actual data. Avoid generalizations that contradict even a single record.
    2. **Semantic Salience**: Focus on relationships between variables that are semantically meaningful (e.g., "women aged 21–43" or "high-income individuals"). Avoid arbitrary correlations.
    3. **Clarity**: Use natural language that is easy to understand. Avoid overly complex nested conditions. Keep conditions to 2–3 attributes max.
    4. **Variety**: Mix different types of statements:
       - Universal claims: "All X satisfy property Y"
       - Conditional claims: "If a person is X, then Y"
       - Existential claims: "There exists at least one X such that Y" — only use these when the combination of X and Y is semantically surprising or noteworthy
       - Majority/frequency claims: "Most X have property Y"
    
    NON-REDUNDANCY RULE — strictly enforce this:
    Before adding a statement, check whether a stricter or more informative version of it is also true. If "All X with A >= 10 have B >= 5" is true, do NOT also include "All X with A >= 10 have B >= 4" — only keep the tightest bound that still holds. Similarly, do not include a statement whose information is fully contained in another statement you've already written.
    
    AVOID:
    - Overly specific conditions that apply to only 1–2 individuals
    - Statements that express the same fact at different levels of precision (keep only the most precise version)
    - Bare existentials with no relational insight (e.g., "There exists a student with GPA 3.4" — this is not interesting on its own)
    - Probabilistic language (e.g., "likely", "probably") unless you have strong statistical support
    
    GOOD EXAMPLES (use these as a guide for style and depth):
    - For all individuals in the table, if the person is a woman, then their age is between 21 and 43 years.
    - There exists at least one man in the table whose resting heart rate is less than 70 bpm.
    - For all individuals with BMI greater than 30, their annual income is less than or equal to $45k.
    - If a person is a man aged over 50, then their BMI is less than 33.
    - All individuals who sleep less than 7 hours per night have a resting HR greater than 70 bpm.
    - Most individuals in the table have an average step count greater than 5000 steps per day.
    - If an individual is a woman aged below 30, then their height is greater than 150 cm.
    - Every individual with a BMI between 20 and 25 has an age less than 40.
    - For all individuals with annual income greater than $70k, their age is less than 50.
    
    BAD EXAMPLES (do not produce statements like these):
    - "There exists at least one individual with a BMI less than 20." — no meaningful relationship
    - "There exists at least one student whose study hours per week are less than 18." — no contrast or condition
    - "There exists at least one student whose age is 17 or less." — trivially true, no insight
    - "There exists at least one student whose extracurricular count is 5." — bare existential, not useful
    
    Here is the data:
    {lines}
    
    Generate your statements below, one per line. Write as many as the data supports, but stop before adding any statement that is redundant with one you've already written."""
        },
    ]
        
        #Generate the statements necessary
        generation = generator(
        prompt1,
        do_sample=False,
        temperature=1.0,
        top_p=1,
        max_new_tokens=10000,
        eos_token_id=tokenizer.eos_token_id)
        # print(f"Generation: {generation[0]['generated_text']}")
        # Get the assistant message from generated_text
        statements_LLM_output = generation[-1]['generated_text'][-1]  # last item
        clean_statements_LLM_output_text = statements_LLM_output['content']  # this is your CSV string
        statements_LLM_output_text = re.sub(r"<think>.*?</think>", "", clean_statements_LLM_output_text, flags=re.DOTALL)
        # Preview
        # print(statements_LLM_output_text[1000:2000])
        #save the output of the LLM generated tasks
        file_name_LLM = f"LLM_statements_{csv_file[:-4]}.txt"
    
        folder_path_LLM_statements = "GPT/b.LLM_Inferences"
    
        full_path_LLM_statements = os.path.join(folder_path_LLM_statements, file_name_LLM)
    
    
        with open(full_path_LLM_statements, "w", encoding="utf-8") as f:
          f.write(statements_LLM_output_text)
        print(f"Saved {file_name_LLM}.")


In [ ]:
import re

batchs = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]

for b in batchs:
    generate(b)

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'max_new_tokens', 'do_sample', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Processing table_0.csv: 15 rows


[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_0.txt.
Processing table_1.csv: 15 rows


[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Saved LLM_statements_table_56.txt.
Processing table_57.csv: 15 rows


[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_57.txt.
Processing table_58.csv: 15 rows


[transformers] Both `max_new_tokens` (=10000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved LLM_statements_table_58.txt.
Processing table_59.csv: 15 rows


In [7]:
import os
import re
import json
import ast
import subprocess
import pandas as pd

# helper to extract code fences if present
def extract_python_code(raw: str) -> str:
    """
    Strip markdown fences if present (```python ... ``` or ``` ... ```).
    Falls back to returning the full string if no fences are found.
    """
    match = re.search(r"```(?:python)?\s*\n(.*?)```", raw, re.DOTALL)
    if match:
        return match.group(1).strip()
    return raw.strip()

def extract_statements_from_txt(raw: str) -> list[str]:
    """
    Extract the statements list from messy model output like:
    assistantfinal{"statements":[...]}
    """
    raw = raw.strip()

    match = re.search(r'(\{\s*"statements"\s*:\s*\[.*?\]\s*\})', raw, re.DOTALL)
    if not match:
        raise ValueError("Could not find a JSON object with a 'statements' field.")

    obj_text = match.group(1)

    try:
        obj = json.loads(obj_text)
    except Exception:
        obj = ast.literal_eval(obj_text)

    statements = obj["statements"]
    if not isinstance(statements, list):
        raise ValueError("'statements' is not a list.")

    return [str(s).strip() for s in statements if str(s).strip()]

def extract_real_python(raw: str) -> str:
    """
    Remove model thinking and keep only the actual Python code.
    Priority:
    1. Content after 'assistantfinal'
    2. Start from first 'import pandas as pd'
    3. Fallback to first import/from line
    """
    raw = raw.strip()

    # Prefer everything after assistantfinal
    m = re.search(r"assistantfinal\s*(.*)$", raw, re.DOTALL)
    if m:
        candidate = m.group(1).strip()
    else:
        candidate = raw

    # Remove markdown fences if any
    candidate = extract_python_code(candidate)

    # Best anchor: code starts at import pandas as pd
    m = re.search(r"(?ms)^import pandas as pd\b.*$", candidate)
    if m:
        return m.group(0).strip()

    # Fallback: start at first import/from line
    m = re.search(r"(?ms)^(?:import|from)\s+.*$", candidate)
    if m:
        return m.group(0).strip()

    return candidate.strip()

LLM_statement_text_files = sorted(
    f for f in os.listdir(folder_path_LLM_statements) if f.endswith(".txt")
)

csv_files = [f for f in os.listdir(folder_path) if f.endswith(".csv")]

for LLM_statement_text_file in LLM_statement_text_files:
    full_path_stmt = os.path.join(folder_path_LLM_statements, LLM_statement_text_file)

    with open(full_path_stmt, "r", encoding="utf-8") as f:
        raw_stmt_text = f.read()

    if not raw_stmt_text.strip():
        print(f"{LLM_statement_text_file} is empty, skipping")
        continue

    try:
        statements = extract_statements_from_txt(raw_stmt_text)
    except Exception as e:
        print(f"{LLM_statement_text_file}: failed to parse statements -> {e}")
        continue

    print(f"\nProcessing {LLM_statement_text_file}.")
    print(f"Parsed {len(statements)} statements.")

    statements_text = "\n".join(
        f"{i+1}. {stmt}" for i, stmt in enumerate(statements)
    )

    matched_csv = None
    for csv_file in csv_files:
        if csv_file[:-4] in LLM_statement_text_file:
            matched_csv = csv_file
            break

    if matched_csv is None:
        print(f"No matching CSV found for {LLM_statement_text_file}, skipping.")
        continue

    full_csv_path = os.path.join(folder_path, matched_csv)
    df = pd.read_csv(full_csv_path)

    prompt2 = [
        {"role": "system", "content": "You are an expert data analyst."},
        {"role": "user", "content": f"""Do NOT repeat the instructions or the code provided.
Only output the requested Python code. Do NOT wrap the code in markdown fences.

Here's your task: Given the following statements:

{statements_text}

and the header names of the table {list(df.columns)}, write a python code (using pandas package)
that checks whether each statement is True or False and prints a justification.

The CSV is already located at: "{full_csv_path}" — hardcode this path directly in the script (no sys.argv).
It should also convert any turn numbers stored as strings into integers.
Everything you output must be valid, immediately runnable Python with no markdown or commentary outside comments.

Here is an example structure to follow:
import pandas as pd

def print_result(statement_no: int, description: str, truth: bool, explanation: str):
    status = "TRUE" if truth else "FALSE"
    print(f"\\nStatement {{statement_no}}: {{status}}")
    print(f"  - {{description}}")
    print(f"  - Explanation: {{explanation}}")

def stmt_1(df: pd.DataFrame):
    \"\"\"1. For all individuals, if the person is a woman, then her age is between 21 and 43.\"\"\"
    women = df[df["gender"] == "F"]
    condition = women["age"].between(21, 43, inclusive="both")
    truth = condition.all()
    if truth:
        expl = f"All {{len(women)}} women are aged 21–43."
    else:
        viol = women[~condition]
        expl = f"{{len(viol)}} women violate the rule (ages: {{', '.join(map(str, viol['age'].tolist()))}})."
    return truth, expl

def main():
    df = pd.read_csv("{full_csv_path}")
    checks = [(1, stmt_1)]  # extend for all statements
    for num, func in checks:
        truth, explanation = func(df)
        print_result(num, func.__doc__.strip(), truth, explanation)

if __name__ == "__main__":
    main()"""}
    ]

    # ── Generate ───────────────────────────────────────────────────────────
    generation = generator(
        prompt2,
        do_sample=False,
        temperature=1.0,
        top_p=1,
        max_new_tokens=100000,
        eos_token_id=tokenizer.eos_token_id,
    )

    python_LLM_output = generation[-1]["generated_text"][-1]
    raw_code = python_LLM_output["content"]

    # ── Clean: keep only actual python code ───────────────────────────────
    python_code = extract_real_python(raw_code)

    # ── Save ───────────────────────────────────────────────────────────────
    python_file_name_LLM = f"python_code_{matched_csv[:-4]}.py"
    full_path_py = os.path.join(folder_path_python_code, python_file_name_LLM)

    with open(full_path_py, "w", encoding="utf-8") as f:
        f.write(python_code)
    print(f"Saved {python_file_name_LLM}")

    # ── Execute automatically ──────────────────────────────────────────────
    print(f"Running {python_file_name_LLM}...")
    result = subprocess.run(
        ["python3", full_path_py],
        capture_output=True,
        text=True,
    )

    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(f"[ERROR] Script exited with code {result.returncode}")
        print(result.stderr)
    else:
        print(f"[OK] {python_file_name_LLM} completed successfully.")

    results_file_name = f"validation_gpt_inferences_{matched_csv[:-4]}.txt"
    full_path_results_file = os.path.join(
        folder_path_python_output_checking_statements,
        results_file_name
    )

    with open(full_path_results_file, "w", encoding="utf-8") as f:
        f.write(result.stdout)
        if result.returncode != 0:
            f.write(f"\n[ERROR] Script exited with code {result.returncode}\n")
            f.write(result.stderr)

    print(f"Saved {results_file_name}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Processing LLM_statements_table_0.txt.
Parsed 8 statements.
Saved python_code_table_0.py
Running python_code_table_0.py...


[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - All students with test scores of 90 or higher are club members.
  - Explanation: All 4 students with test_score ≥ 90 are club members.

Statement 2: TRUE
  - All students who study less than 5 hours per week have test scores of at most 74.
  - Explanation: All 4 low‑study students have test_score ≤ 74.

Statement 3: TRUE
  - All students who study at least 9 hours per week have attendance rates of at least 97%.
  - Explanation: All 4 high‑study students have attendance_rate ≥ 97%.

Statement 4: TRUE
  - All students with attendance rates of at least 95% study at least 8 hours per week.
  - Explanation: All 6 well‑attending students study ≥ 8 hrs/week.

Statement 5: TRUE
  - All students with test scores of at least 85 have attendance rates of at least 94.8%.
  - Explanation: All 7 high‑scoring students have attendance_rate ≥ 94.8%.

Statement 6: TRUE
  - All 12th‑grade students have attendance rates of at least 91.2%.
  - Explanation: All 3 12th‑grade students ha

[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Statement 1: TRUE
  - 1. All organic farms grow soybean.
  - Explanation: All 0 organic farms grow soybean.

Statement 2: FALSE
  - 2. All soybean farms are organic.
  - Explanation: 4 soybean farms are not organic (farm IDs: F003, F007, F011, F015).

Statement 3: TRUE
  - 3. All wheat farms have irrigation between 13.7 and 14.8 hours per week.
  - Explanation: All 4 wheat farms meet the irrigation range.

Statement 4: TRUE
  - 4. All corn farms have irrigation between 16.5 and 17.4 hours per week.
  - Explanation: All 4 corn farms meet the irrigation range.

Statement 5: TRUE
  - 5. All rice farms have irrigation between 19.2 and 20.5 hours per week.
  - Explanation: All 3 rice farms meet the irrigation range.

Statement 6: TRUE
  - 6. All organic farms have a soil quality index of at least 78.1.
  - Explanation: All 0 organic farms have soil_quality_index ≥ 78.1.

Statement 7: TRUE
  - 7. All soybean farms have a yield per acre of at most 2.72 tons.
  - Explanation: All 4 soybean fa

[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - 1. All 5-star hotels have an occupancy rate greater than 85%.
  - Explanation: All 2 5‑star hotels satisfy occupancy_rate > 85%.

Statement 2: TRUE
  - 2. All 3-star hotels have an average nightly rate of at most $133.4.
  - Explanation: All 6 3‑star hotels satisfy avg_nightly_rate ≤ 133.4.

Statement 3: TRUE
  - 3. All hotels with an occupancy rate above 80% have an average nightly rate of at least $176.5.
  - Explanation: All 5 hotels with occupancy_rate > 80% satisfy avg_nightly_rate ≥ 176.5.

Statement 4: TRUE
  - 4. All hotels with a cancellation rate greater than 14% have an average nightly rate of at least $185.
  - Explanation: All 2 hotels with cancellation_rate > 14% satisfy avg_nightly_rate ≥ 185.

Statement 5: TRUE
  - 5. Most hotels have an occupancy rate above 70%.
  - Explanation: 12 out of 15 hotels (80.0%) have occupancy_rate > 70%.

Statement 6: TRUE
  - 6. All 5-star hotels have a cancellation rate greater than 13%.
  - Explanation: All 2 5‑sta

[transformers] Both `max_new_tokens` (=100000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Statement 1: TRUE
  - 1. For all guards, assists per game are at least 5.0.
  - Explanation: All 5 guards have assists_per_game ≥ 5.0.

Statement 2: TRUE
  - 2. All centers have rebounds per game of at least 9.9.
  - Explanation: All 5 centers have rebounds_per_game ≥ 9.9.

Statement 3: TRUE
  - 3. All forwards have points per game of at least 15.6.
  - Explanation: All 5 forwards have points_per_game ≥ 15.6.

Statement 4: TRUE
  - 4. For all players who played at least 74 games, points per game exceed 18.
  - Explanation: All 4 players with ≥74 games have points_per_game > 18.

Statement 5: TRUE
  - 5. Most players have points per game greater than 15.
  - Explanation: 12 out of 15 players (80.0%) have points_per_game > 15.

Statement 6: TRUE
  - 6. All centers have assists per game below 2.5.
  - Explanation: All 5 centers have assists_per_game < 2.5.

Statement 7: TRUE
  - 7. All forwards have rebounds per game at least 6.8.
  - Explanation: All 5 forwards have rebounds_per_game ≥ 